In [ ]:
from pathlib import Path
from typing import Dict, List, Tuple

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import warnings
from rdkit import Chem
from rdkit import RDLogger
from rdkit import DataStructs
from rdkit.Chem import Crippen
from rdkit.Chem import Descriptors
from rdkit.Chem import Lipinski
from rdkit.Chem import MolSurf
from rdkit.Chem import rdPartialCharges
from rdkit.Chem import rdMolDescriptors
from scipy.stats import spearmanr
from sklearn.base import clone
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor
from sklearn.feature_selection import mutual_info_regression
from sklearn.impute import SimpleImputer
from sklearn.linear_model import ElasticNet, Ridge
from sklearn.model_selection import KFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR

try:
    from xgboost import XGBRegressor
except ImportError:
    XGBRegressor = None

warnings.filterwarnings("ignore")
RDLogger.DisableLog("rdApp.warning")

ROOT_CANDIDATES = [
    Path("/home/sy/SY/N-\u4f53\u7cfb"),
    Path("/home/sy/SY/N-体系"),
]
ROOT = None
for _root_candidate in ROOT_CANDIDATES:
    if (_root_candidate / "FE clac.xlsx").exists() or (_root_candidate / "FE_clac.xlsx").exists() or (_root_candidate / "outputs").exists():
        ROOT = _root_candidate
        break
if ROOT is None:
    ROOT = ROOT_CANDIDATES[-1]

INPUT_CANDIDATES = [
    ROOT / "FE clac.xlsx",
    ROOT / "FE_clac.xlsx",
]
SOURCE_OUTPUT_CANDIDATES = [ROOT / "ML" / "outputs", ROOT / "outputs"]
SOURCE_OUTPUT_DIR = None
for _candidate in SOURCE_OUTPUT_CANDIDATES:
    if _candidate.exists():
        SOURCE_OUTPUT_DIR = _candidate
        break
if SOURCE_OUTPUT_DIR is None:
    SOURCE_OUTPUT_DIR = SOURCE_OUTPUT_CANDIDATES[0]

STAGE1_DIR = SOURCE_OUTPUT_DIR / "cell1_feature_selection"
STEPWISE_PLOT_DIR = STAGE1_DIR / "stepwise_plots"
STAGE1_DIR.mkdir(parents=True, exist_ok=True)
STEPWISE_PLOT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_COLUMNS = [
    "FEaverage H2",
    "FEaverage CO",
    "FEaverage CH4",
    "FEaverage C2H4",
    "FE HCOO- ",
    "FE CH3COO- ",
    "FE CH3CH2OH ",
    "FE n-Propanol",
    "FE C2+",
]
BASE_COLUMNS = ["canonical_smiles", "CAS No.", "input_smiles", "E vs RHE"] + TARGET_COLUMNS
SHEET_NAME = "total"
FEATURE_SELECTOR_MODELS = ["ridge", "elastic_net", "svr_rbf", "random_forest"] + (
    ["xgboost"] if XGBRegressor is not None else ["extra_trees"]
)
INNER_SPLITS = 5
RANDOM_STATE = 42
CV_N_JOBS = 30
MODEL_N_JOBS = 1
STEPWISE_PATIENCE = 2
STEPWISE_MIN_IMPROVEMENT = 0.001
MORGAN_RADIUS = 2
MORGAN_NBITS = 128


def target_to_safe_name(target: str) -> str:
    return target.replace("/", "_").replace(" ", "_").replace("+", "plus").replace("-", "minus")


def resolve_input_file() -> Path:
    for path in INPUT_CANDIDATES:
        if path.exists():
            return path
    tried = "\n".join(str(p) for p in INPUT_CANDIDATES)
    raise FileNotFoundError("No Excel input file found. Tried:\n" + tried)


def canonicalize_smiles(smiles: str) -> str:
    if smiles == "Cu":
        return "[Cu]"
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return ""
    return Chem.MolToSmiles(mol)


def load_and_aggregate() -> Tuple[pd.DataFrame, pd.DataFrame]:
    raw = pd.read_excel(resolve_input_file(), sheet_name=SHEET_NAME)
    raw = raw.loc[:, [c for c in raw.columns if pd.notna(c)]].copy()
    raw[["No", "CAS No.", "SMILES"]] = raw[["No", "CAS No.", "SMILES"]].ffill()
    potential_rows = raw[raw["E vs RHE"].notna()].copy()
    potential_rows["SMILES"] = potential_rows["SMILES"].astype(str).str.strip()
    potential_rows = potential_rows[potential_rows["SMILES"] != ""].copy()
    potential_rows["canonical_smiles"] = potential_rows["SMILES"].map(canonicalize_smiles)
    potential_rows = potential_rows[potential_rows["canonical_smiles"] != ""].copy()
    modeling_raw = potential_rows[potential_rows["canonical_smiles"] != "[Cu]"].copy()
    grouped = (
        modeling_raw.groupby("canonical_smiles", dropna=False)
        .agg(
            {
                "CAS No.": lambda s: " | ".join(sorted({str(v) for v in s if pd.notna(v)})),
                "SMILES": "first",
                "E vs RHE": lambda s: " | ".join(sorted({str(v) for v in s if pd.notna(v)})),
                **{target: "max" for target in TARGET_COLUMNS},
            }
        )
        .reset_index()
        .rename(columns={"SMILES": "input_smiles"})
    )
    return potential_rows, grouped


def build_descriptor_table(df: pd.DataFrame) -> pd.DataFrame:
    primary_amine = Chem.MolFromSmarts("[NX3;H2][CX4]")
    secondary_amine = Chem.MolFromSmarts("[NX3;H1]([CX4])[CX4]")
    tertiary_amine = Chem.MolFromSmarts("[NX3;H0]([CX4])([CX4])[CX4]")
    quaternary_ammonium = Chem.MolFromSmarts("[NX4+]")
    hydroxyl = Chem.MolFromSmarts("[OX2H]")
    carbonyl = Chem.MolFromSmarts("[CX3]=[OX1]")
    ether = Chem.MolFromSmarts("[OD2]([#6])[#6]")
    pyridine_ring = Chem.MolFromSmarts("n1ccccc1")
    imidazole_like = Chem.MolFromSmarts("c1ncc[nH]1")
    aromatic_n = Chem.MolFromSmarts("[n]")
    amide_n = Chem.MolFromSmarts("[NX3][CX3](=[OX1])[#6]")
    protonatable_amine_n = Chem.MolFromSmarts("[NX3;!$(NC=O);!$([nH]);!$(N[a])]")

    records: List[Dict[str, float]] = []
    for smiles in df["canonical_smiles"]:
        mol = Chem.MolFromSmiles(smiles)
        row: Dict[str, float] = {"canonical_smiles": smiles}
        for name, func in Descriptors._descList:
            try:
                row[name] = func(mol)
            except Exception:
                row[name] = np.nan

        rdPartialCharges.ComputeGasteigerCharges(mol)
        partial_charges = []
        n_charges = []
        o_charges = []
        for atom in mol.GetAtoms():
            try:
                charge = float(atom.GetProp("_GasteigerCharge"))
            except Exception:
                continue
            if not np.isfinite(charge):
                continue
            partial_charges.append(charge)
            if atom.GetSymbol() == "N":
                n_charges.append(charge)
            elif atom.GetSymbol() == "O":
                o_charges.append(charge)

        atom_symbols = [atom.GetSymbol() for atom in mol.GetAtoms()]
        total_atoms = len(atom_symbols) if atom_symbols else 1
        aromatic_atoms = sum(atom.GetIsAromatic() for atom in mol.GetAtoms())
        sp3_carbons = sum(atom.GetSymbol() == "C" and atom.GetHybridization().name == "SP3" for atom in mol.GetAtoms())
        carbon_count = atom_symbols.count("C")
        nitrogen_count = atom_symbols.count("N")
        oxygen_count = atom_symbols.count("O")
        hetero_count = Lipinski.NumHeteroatoms(mol)
        formal_charge = sum(atom.GetFormalCharge() for atom in mol.GetAtoms())
        positive_atoms = sum(atom.GetFormalCharge() > 0 for atom in mol.GetAtoms())
        negative_atoms = sum(atom.GetFormalCharge() < 0 for atom in mol.GetAtoms())
        ring_sizes = [len(ring) for ring in mol.GetRingInfo().AtomRings()]
        mol_logp = Crippen.MolLogP(mol)
        tpsa = rdMolDescriptors.CalcTPSA(mol)
        labute_asa = MolSurf.LabuteASA(mol)
        amine_count = (
            len(mol.GetSubstructMatches(primary_amine))
            + len(mol.GetSubstructMatches(secondary_amine))
            + len(mol.GetSubstructMatches(tertiary_amine))
            + len(mol.GetSubstructMatches(quaternary_ammonium))
        )
        protonatable_n_matches = mol.GetSubstructMatches(protonatable_amine_n)
        protonatable_n_indices = {match[0] for match in protonatable_n_matches}
        protonatable_n_charges = [
            float(atom.GetProp("_GasteigerCharge"))
            for atom in mol.GetAtoms()
            if atom.GetIdx() in protonatable_n_indices and atom.HasProp("_GasteigerCharge")
        ]

        row.update(
            {
                "num_carbons": carbon_count,
                "num_nitrogens": nitrogen_count,
                "num_oxygens": oxygen_count,
                "num_sulfurs": atom_symbols.count("S"),
                "num_phosphorus": atom_symbols.count("P"),
                "num_halogens": sum(sym in {"F", "Cl", "Br", "I"} for sym in atom_symbols),
                "aromatic_atom_fraction": aromatic_atoms / total_atoms,
                "sp3_carbon_fraction": sp3_carbons / max(carbon_count, 1),
                "formal_charge_sum": formal_charge,
                "num_positive_atoms": positive_atoms,
                "num_negative_atoms": negative_atoms,
                "num_primary_amines": len(mol.GetSubstructMatches(primary_amine)),
                "num_secondary_amines": len(mol.GetSubstructMatches(secondary_amine)),
                "num_tertiary_amines": len(mol.GetSubstructMatches(tertiary_amine)),
                "num_quaternary_ammonium": len(mol.GetSubstructMatches(quaternary_ammonium)),
                "num_hydroxyls": len(mol.GetSubstructMatches(hydroxyl)),
                "num_carbonyls": len(mol.GetSubstructMatches(carbonyl)),
                "num_ethers": len(mol.GetSubstructMatches(ether)),
                "num_rings": rdMolDescriptors.CalcNumRings(mol),
                "num_aromatic_rings": rdMolDescriptors.CalcNumAromaticRings(mol),
                "num_aliphatic_rings": rdMolDescriptors.CalcNumAliphaticRings(mol),
                "num_heteroatoms": hetero_count,
                "num_heavy_atoms": mol.GetNumHeavyAtoms(),
                "num_aromatic_nitrogens": len(mol.GetSubstructMatches(aromatic_n)),
                "num_amide_n": len(mol.GetSubstructMatches(amide_n)),
                "pyridine_ring_flag": int(mol.HasSubstructMatch(pyridine_ring)),
                "imidazole_like_ring_flag": int(mol.HasSubstructMatch(imidazole_like)),
                "num_protonatable_amine_n": len(protonatable_n_indices),
                "max_ring_size": max(ring_sizes) if ring_sizes else 0,
                "min_ring_size": min(ring_sizes) if ring_sizes else 0,
                "n_to_c_ratio": nitrogen_count / max(carbon_count, 1),
                "o_to_n_ratio": oxygen_count / max(nitrogen_count, 1),
                "logp_per_carbon": mol_logp / max(carbon_count, 1),
                "tpsa_per_heteroatom": tpsa / max(hetero_count, 1),
                "asa_per_heavy_atom": labute_asa / max(mol.GetNumHeavyAtoms(), 1),
                "polar_to_hydrophobic_ratio": tpsa / max(abs(mol_logp), 0.1),
                "amine_density": amine_count / max(mol.GetNumHeavyAtoms(), 1),
                "rotatable_bond_density": Lipinski.NumRotatableBonds(mol) / max(mol.GetNumHeavyAtoms(), 1),
                "electronic_gasteiger_mean": float(np.mean(partial_charges)) if partial_charges else 0.0,
                "electronic_gasteiger_std": float(np.std(partial_charges)) if partial_charges else 0.0,
                "electronic_gasteiger_min": float(np.min(partial_charges)) if partial_charges else 0.0,
                "electronic_gasteiger_max": float(np.max(partial_charges)) if partial_charges else 0.0,
                "electronic_positive_charge_sum": float(sum(v for v in partial_charges if v > 0)) if partial_charges else 0.0,
                "electronic_negative_charge_sum": float(sum(v for v in partial_charges if v < 0)) if partial_charges else 0.0,
                "electronic_charge_range": float(np.max(partial_charges) - np.min(partial_charges)) if partial_charges else 0.0,
                "electronic_abs_charge_mean": float(np.mean(np.abs(partial_charges))) if partial_charges else 0.0,
                "electronic_abs_charge_max": float(np.max(np.abs(partial_charges))) if partial_charges else 0.0,
                "electronic_n_charge_mean": float(np.mean(n_charges)) if n_charges else 0.0,
                "electronic_n_charge_min": float(np.min(n_charges)) if n_charges else 0.0,
                "electronic_n_charge_max": float(np.max(n_charges)) if n_charges else 0.0,
                "electronic_protonatable_n_charge_mean": float(np.mean(protonatable_n_charges)) if protonatable_n_charges else 0.0,
                "electronic_protonatable_n_charge_min": float(np.min(protonatable_n_charges)) if protonatable_n_charges else 0.0,
                "electronic_protonatable_n_charge_max": float(np.max(protonatable_n_charges)) if protonatable_n_charges else 0.0,
                "electronic_basicity_proxy_1": float(-np.mean(protonatable_n_charges)) if protonatable_n_charges else 0.0,
                "electronic_basicity_proxy_2": (
                    len(protonatable_n_indices) / max(abs(np.mean(protonatable_n_charges)), 0.05)
                    if protonatable_n_charges
                    else 0.0
                ),
                "electronic_o_charge_mean": float(np.mean(o_charges)) if o_charges else 0.0,
                "electronic_o_charge_min": float(np.min(o_charges)) if o_charges else 0.0,
                "electronic_o_charge_max": float(np.max(o_charges)) if o_charges else 0.0,
            }
        )

        morgan = rdMolDescriptors.GetMorganFingerprintAsBitVect(mol, radius=MORGAN_RADIUS, nBits=MORGAN_NBITS)
        morgan_arr = np.zeros((MORGAN_NBITS,), dtype=int)
        DataStructs.ConvertToNumpyArray(morgan, morgan_arr)
        for bit_idx, bit_value in enumerate(morgan_arr):
            row[f"morgan_bit_{bit_idx}"] = int(bit_value)

        records.append(row)

    desc = pd.DataFrame.from_records(records)
    numeric_cols = [c for c in desc.columns if c != "canonical_smiles"]
    desc[numeric_cols] = desc[numeric_cols].replace([np.inf, -np.inf], np.nan)
    valid_cols = [c for c in numeric_cols if desc[c].notna().all() and desc[c].nunique(dropna=True) > 1]
    return desc[["canonical_smiles"] + valid_cols].copy()


def describe_morgan_environment(mol: Chem.Mol, atom_idx: int, radius: int) -> str:
    bond_indices = Chem.FindAtomEnvironmentOfRadiusN(mol, radius, atom_idx)
    atom_indices = {atom_idx}
    for bond_idx in bond_indices:
        bond = mol.GetBondWithIdx(bond_idx)
        atom_indices.add(bond.GetBeginAtomIdx())
        atom_indices.add(bond.GetEndAtomIdx())
    atom_indices = sorted(atom_indices) if atom_indices else [atom_idx]
    fragment = Chem.MolFragmentToSmiles(
        mol,
        atomsToUse=atom_indices,
        bondsToUse=list(bond_indices),
        rootedAtAtom=atom_idx,
        canonical=True,
    )
    return fragment if fragment else f"atom_env_r{radius}"


def build_descriptor_label_map(smiles_series: pd.Series) -> Dict[str, str]:
    label_map: Dict[str, str] = {}
    bit_examples: Dict[int, Dict[str, int]] = {}
    for smiles in smiles_series.dropna().astype(str):
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            continue
        bit_info: Dict[int, List[Tuple[int, int]]] = {}
        _ = rdMolDescriptors.GetMorganFingerprintAsBitVect(
            mol,
            radius=MORGAN_RADIUS,
            nBits=MORGAN_NBITS,
            bitInfo=bit_info,
        )
        for bit_idx, envs in bit_info.items():
            bit_examples.setdefault(bit_idx, {})
            for atom_idx, radius in envs:
                fragment = describe_morgan_environment(mol, atom_idx, radius)
                bit_examples[bit_idx][fragment] = bit_examples[bit_idx].get(fragment, 0) + 1
    for bit_idx in range(MORGAN_NBITS):
        key = f"morgan_bit_{bit_idx}"
        if bit_idx in bit_examples and bit_examples[bit_idx]:
            label_map[key] = max(bit_examples[bit_idx].items(), key=lambda kv: kv[1])[0]
        else:
            label_map[key] = f"Morgan fragment {bit_idx}"
    return label_map


def deduplicate_labels(labels: List[str]) -> List[str]:
    counts: Dict[str, int] = {}
    out = []
    for label in labels:
        counts[label] = counts.get(label, 0) + 1
        if counts[label] == 1:
            out.append(label)
        else:
            out.append(f"{label} ({counts[label]})")
    return out


def remove_high_correlation(X: pd.DataFrame, threshold: float = 0.999999) -> pd.DataFrame:
    corr = X.corr().abs()
    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    to_drop = [column for column in upper.columns if any(upper[column] > threshold)]
    return X.drop(columns=to_drop)


def prefilter_features(X: pd.DataFrame, y: pd.Series) -> Tuple[pd.DataFrame, pd.DataFrame]:
    X1 = X.loc[:, X.nunique(dropna=True) > 1].copy()
    X2 = remove_high_correlation(X1, threshold=0.999999)
    mi = mutual_info_regression(X2, y, random_state=RANDOM_STATE)
    spearman_scores = []
    for col in X2.columns:
        coef = spearmanr(X2[col], y).correlation
        spearman_scores.append(0.0 if pd.isna(coef) else abs(coef))
    score_df = pd.DataFrame({"descriptor": X2.columns, "mutual_info": mi, "abs_spearman": spearman_scores})
    score_df["combined_score"] = score_df["mutual_info"].rank(method="average", ascending=False) + score_df[
        "abs_spearman"
    ].rank(method="average", ascending=False)
    score_df = score_df.sort_values(["combined_score", "mutual_info", "abs_spearman"], ascending=[True, False, False])
    return X2.copy(), score_df


def build_feature_selector_specs() -> Dict[str, Pipeline]:
    specs = {
        "ridge": Pipeline(
            [("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler()), ("model", Ridge(alpha=1.0))]
        ),
        "elastic_net": Pipeline(
            [
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
                ("model", ElasticNet(alpha=0.01, l1_ratio=0.5, max_iter=100000, tol=1e-3, random_state=42)),
            ]
        ),
        "svr_rbf": Pipeline(
            [
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
                ("model", SVR(kernel="rbf", C=10.0, gamma="scale", epsilon=0.1)),
            ]
        ),
        "random_forest": Pipeline(
            [
                ("imputer", SimpleImputer(strategy="median")),
                (
                    "model",
                    RandomForestRegressor(
                        n_estimators=300,
                        max_depth=None,
                        min_samples_leaf=1,
                        max_features="sqrt",
                        random_state=42,
                        n_jobs=MODEL_N_JOBS,
                    ),
                ),
            ]
        ),
        "extra_trees": Pipeline(
            [
                ("imputer", SimpleImputer(strategy="median")),
                (
                    "model",
                    ExtraTreesRegressor(
                        n_estimators=300,
                        max_depth=None,
                        min_samples_leaf=1,
                        max_features="sqrt",
                        random_state=42,
                        n_jobs=MODEL_N_JOBS,
                    ),
                ),
            ]
        ),
    }
    if XGBRegressor is not None:
        specs["xgboost"] = Pipeline(
            [
                ("imputer", SimpleImputer(strategy="median")),
                (
                    "model",
                    XGBRegressor(
                        n_estimators=300,
                        max_depth=4,
                        learning_rate=0.05,
                        subsample=0.85,
                        colsample_bytree=0.85,
                        objective="reg:squarederror",
                        random_state=42,
                        n_jobs=MODEL_N_JOBS,
                        verbosity=0,
                    ),
                ),
            ]
        )
    return specs


def cv_r2_for_features(X: pd.DataFrame, y: pd.Series, cv, selector_model_name: str) -> float:
    model = clone(build_feature_selector_specs()[selector_model_name])
    scores = cross_val_score(model, X, y, cv=cv, scoring="r2", n_jobs=CV_N_JOBS)
    return float(np.mean(scores))


def forward_stepwise_selection(
    X: pd.DataFrame, y: pd.Series, cv, selector_model_name: str, label_map: Dict[str, str] | None = None
) -> Tuple[List[str], pd.DataFrame]:
    remaining = list(X.columns)
    current_selected: List[str] = []
    history: List[Dict[str, float]] = []
    best_overall_score = -np.inf
    best_selected: List[str] = []
    no_improvement_rounds = 0
    while remaining:
        step_no = len(current_selected) + 1
        print(f"  [Feature Selection] selector={selector_model_name} step {step_no}: evaluating {len(remaining)} features")
        round_results = []
        for feature in remaining:
            trial = current_selected + [feature]
            score = cv_r2_for_features(X[trial], y, cv=cv, selector_model_name=selector_model_name)
            round_results.append((feature, score))
        round_results.sort(key=lambda x: x[1], reverse=True)
        best_feature, candidate_score = round_results[0]
        prev_score = history[-1]["cv_r2_after_add"] if history else np.nan
        improvement = candidate_score - prev_score if np.isfinite(prev_score) else np.nan
        current_selected.append(best_feature)
        remaining.remove(best_feature)
        history.append(
            {
                "step": len(current_selected),
                "added_feature": best_feature,
                "cv_r2_after_add": candidate_score,
                "improvement": improvement,
                "selected_features": " | ".join(current_selected),
                "selected_features_readable": " | ".join(label_map.get(f, f) if label_map else f for f in current_selected),
            }
        )
        if candidate_score > best_overall_score + STEPWISE_MIN_IMPROVEMENT:
            best_overall_score = candidate_score
            best_selected = current_selected.copy()
            no_improvement_rounds = 0
        else:
            no_improvement_rounds += 1
            if no_improvement_rounds >= STEPWISE_PATIENCE:
                break
    history_df = pd.DataFrame(history)
    if not history_df.empty:
        history_df["is_best_step"] = False
        best_idx = history_df["cv_r2_after_add"].idxmax()
        history_df.loc[best_idx, "is_best_step"] = True
        history_df["selector_model"] = selector_model_name
    return best_selected, history_df


def make_stepwise_plot(step_history: pd.DataFrame, target: str, selector_model: str) -> Path:
    plot_df = step_history.copy().sort_values("step").reset_index(drop=True)
    y = plot_df["cv_r2_after_add"].astype(float).to_numpy()
    finite_y = y[np.isfinite(y)]
    if len(finite_y) == 0:
        finite_y = np.array([0.0])
    q1 = float(np.quantile(finite_y, 0.25))
    q3 = float(np.quantile(finite_y, 0.75))
    iqr = q3 - q1
    lower = q1 - 3.0 * iqr
    upper = q3 + 3.0 * iqr
    plot_df["cv_r2_display"] = np.clip(y, lower, upper)
    best_row = plot_df.loc[plot_df["cv_r2_after_add"].idxmax()]
    plot_path = STEPWISE_PLOT_DIR / f"{target_to_safe_name(target)}_{selector_model}_stepwise.png"
    plt.figure(figsize=(8.5, 5.5))
    plt.plot(plot_df["step"], plot_df["cv_r2_display"], marker="o", linewidth=1.8, color="#4C78A8")
    plt.scatter([best_row["step"]], [best_row["cv_r2_display"]], color="#F28E2B", s=90, zorder=4, label="best step")
    plt.annotate(
        f"best: step {int(best_row['step'])}\nCV R2={best_row['cv_r2_after_add']:.3f}",
        xy=(best_row["step"], best_row["cv_r2_display"]),
        xytext=(8, 10),
        textcoords="offset points",
        fontsize=9,
        bbox=dict(boxstyle="round,pad=0.25", fc="white", ec="#999999", alpha=0.9),
    )
    plt.xlabel("Number of Selected Features")
    plt.ylabel("CV R2")
    plt.title(f"Stepwise Feature Selection: {target} [{selector_model}]")
    plt.grid(alpha=0.25)
    plt.legend(frameon=False, loc="best")
    plt.tight_layout()
    plt.savefig(plot_path, dpi=300, bbox_inches="tight")
    plt.close()
    return plot_path


print(f"[Cell1] Loading raw Excel from {resolve_input_file()}")
_, aggregated_df = load_and_aggregate()
descriptor_df = build_descriptor_table(aggregated_df)
modeling_df = aggregated_df.merge(descriptor_df, on="canonical_smiles", how="inner")
descriptor_columns = [c for c in modeling_df.columns if c not in BASE_COLUMNS]
descriptor_label_map = build_descriptor_label_map(modeling_df["canonical_smiles"])
selector_cv = KFold(n_splits=INNER_SPLITS, shuffle=True, random_state=RANDOM_STATE)
selector_catalog_rows = []
selection_rows = []
filter_rows = []
summary_lines = [
    "# Cell 1 Feature Selection Summary",
    "",
    f"- Source descriptor matrix: `{SOURCE_OUTPUT_DIR / 'co2rr_ml_descriptor_matrix.csv'}`",
    f"- Selector models: {', '.join(FEATURE_SELECTOR_MODELS)}",
    f"- CV_n_jobs: {CV_N_JOBS}",
    "",
]

for target in TARGET_COLUMNS:
    print(f"[Cell1] ===== Processing target: {target} =====")
    target_df = modeling_df[["canonical_smiles", target] + descriptor_columns].copy()
    target_df = target_df[target_df[target].notna()].reset_index(drop=True)
    X_all = target_df[descriptor_columns].copy()
    y_all = target_df[target].astype(float).reset_index(drop=True)
    X_prefiltered, filter_scores = prefilter_features(X_all, y_all)
    filter_scores.insert(0, "target", target)
    filter_rows.append(filter_scores)
    fallback_features = X_prefiltered.columns[: min(5, X_prefiltered.shape[1])].tolist()

    for selector_model in FEATURE_SELECTOR_MODELS:
        print(f"[Cell1] {target} | selector={selector_model}")
        selected_features, step_history = forward_stepwise_selection(
            X_prefiltered,
            y_all,
            cv=selector_cv,
            selector_model_name=selector_model,
            label_map=descriptor_label_map,
        )
        if not selected_features:
            selected_features = fallback_features.copy()
            step_history = pd.DataFrame(
                [
                    {
                        "step": i + 1,
                        "added_feature": feat,
                        "cv_r2_after_add": np.nan,
                        "improvement": np.nan,
                        "selected_features": " | ".join(selected_features[: i + 1]),
                        "selected_features_readable": " | ".join(
                            descriptor_label_map.get(f, f) for f in selected_features[: i + 1]
                        ),
                        "is_best_step": i + 1 == len(selected_features),
                        "selector_model": selector_model,
                    }
                    for i, feat in enumerate(selected_features)
                ]
            )
        best_row = step_history.loc[step_history["cv_r2_after_add"].idxmax()] if not step_history.empty else None
        step_history.insert(0, "target", target)
        step_history["added_feature_readable"] = step_history["added_feature"].map(
            lambda x: descriptor_label_map.get(x, x)
        )
        selection_rows.append(step_history)
        stepwise_path = make_stepwise_plot(step_history.drop(columns=["target"], errors="ignore"), target, selector_model)
        selector_catalog_rows.append(
            {
                "target": target,
                "selector_model": selector_model,
                "selector_best_step": int(best_row["step"]) if best_row is not None else len(selected_features),
                "selector_best_cv_r2": float(best_row["cv_r2_after_add"]) if best_row is not None else np.nan,
                "n_samples": len(target_df),
                "n_prefiltered_features": X_prefiltered.shape[1],
                "n_selected_features": len(selected_features),
                "selected_features": " | ".join(selected_features),
                "selected_features_readable": " | ".join(descriptor_label_map.get(f, f) for f in selected_features),
                "stepwise_plot": str(stepwise_path),
            }
        )
        summary_lines.append(
            f"- `{target}` | selector=`{selector_model}` | best step={selector_catalog_rows[-1]['selector_best_step']} | "
            f"best CV R2={selector_catalog_rows[-1]['selector_best_cv_r2']:.3f} | n_features={len(selected_features)}"
        )

selector_catalog_df = pd.DataFrame(selector_catalog_rows)
selection_df = pd.concat(selection_rows, ignore_index=True)
filter_df = pd.concat(filter_rows, ignore_index=True)

selector_catalog_df.to_csv(STAGE1_DIR / "cell1_selector_feature_catalog.csv", index=False)
selection_df.to_csv(STAGE1_DIR / "cell1_feature_selection_steps.csv", index=False)
filter_df.to_csv(STAGE1_DIR / "cell1_filter_scores.csv", index=False)
modeling_df.to_csv(STAGE1_DIR / "cell1_descriptor_matrix.csv", index=False)

with pd.ExcelWriter(STAGE1_DIR / "cell1_feature_selection_results.xlsx", engine="openpyxl") as writer:
    selector_catalog_df.to_excel(writer, sheet_name="selector_catalog", index=False)
    selection_df.to_excel(writer, sheet_name="feature_selection_steps", index=False)
    filter_df.to_excel(writer, sheet_name="filter_scores", index=False)
    modeling_df.to_excel(writer, sheet_name="descriptor_matrix", index=False)

(STAGE1_DIR / "cell1_feature_selection_summary.md").write_text("\n".join(summary_lines), encoding="utf-8")
print(f"[Cell1] Done. Outputs written to {STAGE1_DIR}")


In [1]:
from pathlib import Path
from typing import Any, Dict, List, Tuple

import optuna
import shap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from rdkit import Chem
from rdkit.Chem import rdMolDescriptors
from sklearn.base import clone
from sklearn.cross_decomposition import PLSRegression
from sklearn.ensemble import AdaBoostRegressor, ExtraTreesRegressor, GradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import ElasticNet, Lasso, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, cross_val_predict, cross_val_score
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR

try:
    from xgboost import XGBRegressor
except ImportError:
    XGBRegressor = None

try:
    from lightgbm import LGBMRegressor
except ImportError:
    LGBMRegressor = None

try:
    from catboost import CatBoostRegressor
except ImportError:
    CatBoostRegressor = None

ROOT_CANDIDATES = [
    Path("/home/sy/SY/N-\u4f53\u7cfb"),
    Path("/home/sy/SY/N-浣撶郴"),
]
ROOT = None
for _root_candidate in ROOT_CANDIDATES:
    if (_root_candidate / "FE clac.xlsx").exists() or (_root_candidate / "FE_clac.xlsx").exists() or (_root_candidate / "outputs").exists():
        ROOT = _root_candidate
        break
if ROOT is None:
    ROOT = ROOT_CANDIDATES[0]

SOURCE_OUTPUT_CANDIDATES = [ROOT / "ML" / "outputs", ROOT / "outputs"]
SOURCE_OUTPUT_DIR = None
for _candidate in SOURCE_OUTPUT_CANDIDATES:
    if _candidate.exists():
        SOURCE_OUTPUT_DIR = _candidate
        break
if SOURCE_OUTPUT_DIR is None:
    SOURCE_OUTPUT_DIR = SOURCE_OUTPUT_CANDIDATES[-1]

STAGE1_DIR = SOURCE_OUTPUT_DIR / "cell1_feature_selection"
TARGET_COLUMNS = [
    "FEaverage H2",
    "FEaverage CO",
    "FEaverage CH4",
    "FEaverage C2H4",
    "FE HCOO- ",
    "FE CH3COO- ",
    "FE CH3CH2OH ",
    "FE n-Propanol",
    "FE C2+",
]
BASE_COLUMNS = ["canonical_smiles", "CAS No.", "input_smiles", "E vs RHE"] + TARGET_COLUMNS
INNER_SPLITS = 5
RANDOM_STATE = 42
MORGAN_RADIUS = 2
MORGAN_NBITS = 128

STAGE2_DIR = SOURCE_OUTPUT_DIR / "cell2_model_comparison_from_cell1-20260612"
SHAP_PLOT_DIR = STAGE2_DIR / "shap_plots"
FIT_PLOT_DIR = STAGE2_DIR / "fit_plots"
RESIDUAL_PLOT_DIR = STAGE2_DIR / "residual_plots"
IMPORTANCE_PLOT_DIR = STAGE2_DIR / "importance_plots"
for path in [STAGE2_DIR, SHAP_PLOT_DIR, FIT_PLOT_DIR, RESIDUAL_PLOT_DIR, IMPORTANCE_PLOT_DIR]:
    path.mkdir(parents=True, exist_ok=True)

OUTER_SPLITS = 5
OPTUNA_TRIALS = 30
MODEL_N_JOBS = 1
CV_N_JOBS = 30
PLOT_FONT_FAMILY = "Liberation Sans"
PLOT_FONT_SIZE = 15


def target_to_safe_name(target: str) -> str:
    return target.replace("/", "_").replace(" ", "_").replace("+", "plus").replace("-", "minus")


def describe_morgan_environment(mol: Chem.Mol, atom_idx: int, radius: int) -> str:
    bond_indices = Chem.FindAtomEnvironmentOfRadiusN(mol, radius, atom_idx)
    atom_indices = {atom_idx}
    for bond_idx in bond_indices:
        bond = mol.GetBondWithIdx(bond_idx)
        atom_indices.add(bond.GetBeginAtomIdx())
        atom_indices.add(bond.GetEndAtomIdx())
    atom_indices = sorted(atom_indices) if atom_indices else [atom_idx]
    fragment = Chem.MolFragmentToSmiles(
        mol,
        atomsToUse=atom_indices,
        bondsToUse=list(bond_indices),
        rootedAtAtom=atom_idx,
        canonical=True,
    )
    return fragment if fragment else f"atom_env_r{radius}"


def build_descriptor_label_map(smiles_series: pd.Series) -> Dict[str, str]:
    label_map: Dict[str, str] = {}
    bit_examples: Dict[int, Dict[str, int]] = {}
    for smiles in smiles_series.dropna().astype(str):
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            continue
        bit_info: Dict[int, List[Tuple[int, int]]] = {}
        _ = rdMolDescriptors.GetMorganFingerprintAsBitVect(
            mol,
            radius=MORGAN_RADIUS,
            nBits=MORGAN_NBITS,
            bitInfo=bit_info,
        )
        for bit_idx, envs in bit_info.items():
            bit_examples.setdefault(bit_idx, {})
            for atom_idx, radius in envs:
                fragment = describe_morgan_environment(mol, atom_idx, radius)
                bit_examples[bit_idx][fragment] = bit_examples[bit_idx].get(fragment, 0) + 1
    for bit_idx in range(MORGAN_NBITS):
        key = f"morgan_bit_{bit_idx}"
        if bit_idx in bit_examples and bit_examples[bit_idx]:
            label_map[key] = max(bit_examples[bit_idx].items(), key=lambda kv: kv[1])[0]
        else:
            label_map[key] = f"Morgan fragment {bit_idx}"
    return label_map


def deduplicate_labels(labels: List[str]) -> List[str]:
    counts: Dict[str, int] = {}
    out = []
    for label in labels:
        counts[label] = counts.get(label, 0) + 1
        if counts[label] == 1:
            out.append(label)
        else:
            out.append(f"{label} ({counts[label]})")
    return out


def build_model_specs() -> Dict[str, Pipeline]:
    specs = {
        "ridge": Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler()), ("model", Ridge())]),
        "lasso": Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler()), ("model", Lasso(max_iter=100000, tol=1e-3, random_state=42))]),
        "elastic_net": Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler()), ("model", ElasticNet(max_iter=100000, tol=1e-3, random_state=42))]),
        "pls": Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler()), ("model", PLSRegression())]),
        "svr_rbf": Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler()), ("model", SVR(kernel="rbf"))]),
        "knn": Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler()), ("model", KNeighborsRegressor(n_jobs=MODEL_N_JOBS))]),
        "random_forest": Pipeline([("imputer", SimpleImputer(strategy="median")), ("model", RandomForestRegressor(random_state=42, n_jobs=MODEL_N_JOBS))]),
        "extra_trees": Pipeline([("imputer", SimpleImputer(strategy="median")), ("model", ExtraTreesRegressor(random_state=42, n_jobs=MODEL_N_JOBS))]),
        "gradient_boosting": Pipeline([("imputer", SimpleImputer(strategy="median")), ("model", GradientBoostingRegressor(random_state=42))]),
        "adaboost": Pipeline([("imputer", SimpleImputer(strategy="median")), ("model", AdaBoostRegressor(random_state=42))]),
    }
    if XGBRegressor is not None:
        specs["xgboost"] = Pipeline([("imputer", SimpleImputer(strategy="median")), ("model", XGBRegressor(random_state=42, objective="reg:squarederror", n_jobs=MODEL_N_JOBS, verbosity=0))])
    if LGBMRegressor is not None:
        specs["lightgbm"] = Pipeline([("imputer", SimpleImputer(strategy="median")), ("model", LGBMRegressor(random_state=42, verbosity=-1, n_jobs=MODEL_N_JOBS))])
    if CatBoostRegressor is not None:
        specs["catboost"] = Pipeline([("imputer", SimpleImputer(strategy="median")), ("model", CatBoostRegressor(random_state=42, verbose=0, allow_writing_files=False, thread_count=MODEL_N_JOBS))])
    return specs


def suggest_model_params(trial: optuna.Trial, model_name: str, n_features: int) -> Dict[str, object]:
    if model_name == "ridge":
        return {"model__alpha": trial.suggest_float("model__alpha", 1e-2, 1e2, log=True)}
    if model_name == "lasso":
        return {"model__alpha": trial.suggest_float("model__alpha", 1e-3, 1.0, log=True)}
    if model_name == "elastic_net":
        return {"model__alpha": trial.suggest_float("model__alpha", 1e-3, 1.0, log=True), "model__l1_ratio": trial.suggest_float("model__l1_ratio", 0.1, 0.9)}
    if model_name == "pls":
        return {"model__n_components": trial.suggest_int("model__n_components", 2, max(2, min(6, n_features)))}
    if model_name == "svr_rbf":
        return {"model__C": trial.suggest_float("model__C", 1e-1, 1e2, log=True), "model__gamma": trial.suggest_float("model__gamma", 1e-3, 1.0, log=True), "model__epsilon": trial.suggest_float("model__epsilon", 1e-2, 0.5, log=True)}
    if model_name == "knn":
        return {"model__n_neighbors": trial.suggest_int("model__n_neighbors", 3, min(15, max(3, n_features + 2)), step=2), "model__weights": trial.suggest_categorical("model__weights", ["uniform", "distance"]), "model__p": trial.suggest_categorical("model__p", [1, 2])}
    if model_name in {"random_forest", "extra_trees"}:
        return {"model__n_estimators": trial.suggest_int("model__n_estimators", 200, 800, step=200), "model__max_depth": trial.suggest_categorical("model__max_depth", [None, 4, 6, 8]), "model__min_samples_leaf": trial.suggest_categorical("model__min_samples_leaf", [1, 2, 4]), "model__max_features": trial.suggest_categorical("model__max_features", ["sqrt", 0.5, 0.8])}
    if model_name == "gradient_boosting":
        return {"model__n_estimators": trial.suggest_int("model__n_estimators", 100, 400, step=100), "model__learning_rate": trial.suggest_float("model__learning_rate", 0.01, 0.1, log=True), "model__max_depth": trial.suggest_int("model__max_depth", 2, 4), "model__subsample": trial.suggest_float("model__subsample", 0.7, 1.0)}
    if model_name == "adaboost":
        return {"model__n_estimators": trial.suggest_int("model__n_estimators", 100, 400, step=100), "model__learning_rate": trial.suggest_float("model__learning_rate", 0.01, 1.0, log=True), "model__loss": trial.suggest_categorical("model__loss", ["linear", "square", "exponential"])}
    if model_name == "xgboost":
        return {"model__n_estimators": trial.suggest_int("model__n_estimators", 200, 800, step=200), "model__max_depth": trial.suggest_int("model__max_depth", 2, 6), "model__learning_rate": trial.suggest_float("model__learning_rate", 0.01, 0.1, log=True), "model__subsample": trial.suggest_float("model__subsample", 0.7, 1.0), "model__colsample_bytree": trial.suggest_float("model__colsample_bytree", 0.5, 1.0)}
    if model_name == "lightgbm":
        return {"model__n_estimators": trial.suggest_int("model__n_estimators", 200, 800, step=200), "model__max_depth": trial.suggest_categorical("model__max_depth", [-1, 3, 5, 7]), "model__learning_rate": trial.suggest_float("model__learning_rate", 0.01, 0.1, log=True), "model__subsample": trial.suggest_float("model__subsample", 0.7, 1.0), "model__colsample_bytree": trial.suggest_float("model__colsample_bytree", 0.5, 1.0)}
    if model_name == "catboost":
        return {"model__iterations": trial.suggest_int("model__iterations", 200, 800, step=200), "model__depth": trial.suggest_int("model__depth", 3, 8), "model__learning_rate": trial.suggest_float("model__learning_rate", 0.01, 0.1, log=True), "model__subsample": trial.suggest_float("model__subsample", 0.7, 1.0)}
    raise ValueError(f"Unsupported model for Optuna search: {model_name}")


def optimize_single_model(model_name: str, pipeline: Pipeline, X: pd.DataFrame, y: pd.Series, inner_cv, n_trials: int = OPTUNA_TRIALS) -> Tuple[Dict[str, object], Pipeline]:
    print(f"    [Optuna] Start tuning model: {model_name} with {n_trials} trials")
    def objective(trial: optuna.Trial) -> float:
        params = suggest_model_params(trial, model_name, X.shape[1])
        estimator = clone(pipeline)
        estimator.set_params(**params)
        scores = cross_val_score(estimator, X, y, cv=inner_cv, scoring="r2", n_jobs=CV_N_JOBS)
        return float(np.mean(scores))
    study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)
    best_estimator = clone(pipeline)
    best_estimator.set_params(**study.best_params)
    best_estimator.fit(X, y)
    return study.best_params, best_estimator


def tune_and_compare_models(X: pd.DataFrame, y: pd.Series, inner_cv, n_trials: int = OPTUNA_TRIALS) -> Tuple[pd.DataFrame, Dict[str, Pipeline]]:
    result_rows = []
    fitted_estimators: Dict[str, Pipeline] = {}
    model_specs = build_model_specs()
    print(f"  [Model Search] Start comparing {len(model_specs)} models on {X.shape[1]} selected features")
    for model_name, pipeline in model_specs.items():
        best_params, fitted_estimator = optimize_single_model(model_name, pipeline, X, y, inner_cv=inner_cv, n_trials=n_trials)
        pred = cross_val_predict(clone(fitted_estimator), X, y, cv=inner_cv, n_jobs=CV_N_JOBS)
        row = {"model_name": model_name, "cv_r2": r2_score(y, pred), "cv_rmse": mean_squared_error(y, pred, squared=False), "cv_mae": mean_absolute_error(y, pred), "best_params": best_params}
        result_rows.append(row)
        fitted_estimators[model_name] = fitted_estimator
        print(f"  [Model Search] {model_name}: CV R2={row['cv_r2']:.4f}, RMSE={row['cv_rmse']:.4f}, MAE={row['cv_mae']:.4f}")
    result_df = pd.DataFrame(result_rows).sort_values(["cv_r2", "cv_rmse"], ascending=[False, True]).reset_index(drop=True)
    return result_df, fitted_estimators


def transform_feature_matrix_for_model(fitted: Pipeline, X: pd.DataFrame) -> pd.DataFrame:
    transformed = fitted.named_steps["imputer"].transform(X)
    feature_df = pd.DataFrame(transformed, columns=X.columns, index=X.index)
    if "scaler" in fitted.named_steps:
        transformed = fitted.named_steps["scaler"].transform(feature_df)
        feature_df = pd.DataFrame(transformed, columns=X.columns, index=X.index)
    return feature_df


def compute_shap_values(model_name: str, fitted: Pipeline, X: pd.DataFrame) -> Tuple[pd.DataFrame, np.ndarray]:
    feature_df = transform_feature_matrix_for_model(fitted, X)
    estimator = fitted.named_steps["model"]
    if model_name in {"ridge", "lasso", "elastic_net", "pls"}:
        explainer = shap.Explainer(estimator, feature_df)
        shap_values = explainer(feature_df).values
    else:
        try:
            explainer = shap.TreeExplainer(estimator)
            shap_values = explainer.shap_values(feature_df)
        except Exception:
            background = shap.sample(feature_df, min(50, len(feature_df)), random_state=RANDOM_STATE)
            explainer = shap.Explainer(estimator.predict, background)
            shap_values = explainer(feature_df).values
    shap_values = np.asarray(shap_values)
    if shap_values.ndim == 3:
        shap_values = shap_values[..., 0]
    return feature_df, shap_values


def aggregate_fold_shap_results(shap_feature_frames: List[pd.DataFrame], shap_value_arrays: List[np.ndarray]) -> Tuple[pd.DataFrame, np.ndarray]:
    all_columns: List[str] = []
    for df in shap_feature_frames:
        for col in df.columns:
            if col not in all_columns:
                all_columns.append(col)
    aligned_frames = []
    aligned_values = []
    for df, values in zip(shap_feature_frames, shap_value_arrays):
        aligned_frames.append(df.reindex(columns=all_columns, fill_value=0.0))
        value_df = pd.DataFrame(values, columns=df.columns, index=df.index).reindex(columns=all_columns, fill_value=0.0)
        aligned_values.append(value_df.to_numpy())
    return pd.concat(aligned_frames, ignore_index=True), np.vstack(aligned_values)


def build_shap_importance_df(feature_df: pd.DataFrame, shap_values: np.ndarray, label_map: Dict[str, str] | None = None) -> pd.DataFrame:
    mean_abs_shap = np.mean(np.abs(shap_values), axis=0)
    out = pd.DataFrame({"descriptor_raw": feature_df.columns, "importance": mean_abs_shap, "abs_importance": mean_abs_shap})
    out["descriptor"] = out["descriptor_raw"].map(lambda x: label_map.get(x, x) if label_map else x)
    out["descriptor"] = deduplicate_labels(out["descriptor"].tolist())
    return out.sort_values("abs_importance", ascending=False).reset_index(drop=True)


def build_shap_export_df(
    feature_df: pd.DataFrame,
    shap_values: np.ndarray,
    metadata_df: pd.DataFrame,
    target: str,
    selector_model: str,
    best_model_name: str,
    label_map: Dict[str, str],
) -> pd.DataFrame:
    descriptor_labels = deduplicate_labels([label_map.get(c, c) for c in feature_df.columns])
    shap_df = pd.DataFrame(shap_values, columns=descriptor_labels)
    shap_df = shap_df.add_prefix("shap__")
    value_df = feature_df.copy()
    value_df.columns = [f"feature__{label}" for label in descriptor_labels]
    export_df = pd.concat([metadata_df.reset_index(drop=True), value_df.reset_index(drop=True), shap_df.reset_index(drop=True)], axis=1)
    export_df.insert(0, "best_model", best_model_name)
    export_df.insert(0, "selector_model", selector_model)
    export_df.insert(0, "target", target)
    return export_df


def make_shap_plot_from_aggregated(feature_df: pd.DataFrame, shap_values: np.ndarray, target: str, selector_model: str, best_model_name: str, label_map: Dict[str, str]) -> Path:
    plot_df = feature_df.copy()
    labels = [label_map.get(c, c) for c in plot_df.columns]
    plot_df.columns = deduplicate_labels(labels)
    plot_path = SHAP_PLOT_DIR / f"{target_to_safe_name(target)}_{selector_model}_{best_model_name}_shap.png"
    plt.rcParams.update({
        "font.family": PLOT_FONT_FAMILY,
        "font.size": PLOT_FONT_SIZE,
        "font.weight": "bold",
        "axes.labelweight": "bold",
        "axes.titleweight": "bold",
        "axes.labelsize": PLOT_FONT_SIZE,
        "axes.titlesize": PLOT_FONT_SIZE,
        "xtick.labelsize": PLOT_FONT_SIZE,
        "ytick.labelsize": PLOT_FONT_SIZE,
    })
    plt.figure(figsize=(10, 6))
    shap.summary_plot(shap_values, plot_df, show=False, max_display=min(12, plot_df.shape[1]))
    ax = plt.gca()
    ax.set_title(f"SHAP Summary: {target} [{selector_model} -> {best_model_name}]", fontweight="bold", fontsize=PLOT_FONT_SIZE)
    ax.set_xlabel(ax.get_xlabel(), fontweight="bold", fontsize=PLOT_FONT_SIZE)
    ax.spines["bottom"].set_linewidth(1.8)
    ax.tick_params(axis="x", width=1.8, length=5)
    ax.tick_params(axis="y", width=1.2, length=0)
    for label in ax.get_xticklabels():
        label.set_fontweight("bold")
        label.set_fontsize(PLOT_FONT_SIZE)
    for label in ax.get_yticklabels():
        label.set_fontweight("bold")
        label.set_fontsize(PLOT_FONT_SIZE)
    plt.tight_layout()
    plt.savefig(plot_path, dpi=300, bbox_inches="tight")
    plt.close()
    return plot_path


def make_fit_plot(y_true: pd.Series, y_pred: np.ndarray, target: str, selector_model: str, best_model_name: str, stats_text: str) -> Path:
    plot_path = FIT_PLOT_DIR / f"{target_to_safe_name(target)}_{selector_model}_{best_model_name}_fit.png"
    plt.rcParams.update({
        "font.family": PLOT_FONT_FAMILY,
        "font.style": "normal",
        "font.size": PLOT_FONT_SIZE,
        "font.weight": "bold",
        "mathtext.default": "regular",
        "axes.labelweight": "bold",
        "axes.titleweight": "bold",
        "axes.labelsize": PLOT_FONT_SIZE,
        "axes.titlesize": PLOT_FONT_SIZE,
        "xtick.labelsize": PLOT_FONT_SIZE,
        "ytick.labelsize": PLOT_FONT_SIZE,
    })
    x = np.asarray(y_true, dtype=float)
    y = np.asarray(y_pred, dtype=float)
    xy_min = float(min(x.min(), y.min()))
    xy_max = float(max(x.max(), y.max()))
    pad = 0.05 * (xy_max - xy_min if xy_max > xy_min else 1.0)
    fig, ax = plt.subplots(figsize=(6.0, 5.4), dpi=300)
    ax.scatter(
        x,
        y,
        s=50,
        color="#2F7FD1",
        alpha=0.85,
        edgecolor="white",
        linewidth=0.6,
    )
    ax.plot(
        [xy_min - pad, xy_max + pad],
        [xy_min - pad, xy_max + pad],
        linestyle="--",
        color="gray",
        linewidth=1.6,
    )
    ax.set_xlim(xy_min - pad, xy_max + pad)
    ax.set_ylim(xy_min - pad, xy_max + pad)
    ax.set_xlabel("True", fontweight="bold", fontsize=PLOT_FONT_SIZE, fontstyle="normal")
    ax.set_ylabel("Predicted", fontweight="bold", fontsize=PLOT_FONT_SIZE, fontstyle="normal")
    ax.set_title(
        f"Cross-validated Model Performance for {target_to_safe_name(target)}",
        pad=10,
        fontweight="bold",
        fontsize=PLOT_FONT_SIZE,
        fontstyle="normal",
    )
    ax.grid(True, color="#E6EEF5", linewidth=0.8)
    ax.set_axisbelow(True)
    for spine in ["top", "right", "left", "bottom"]:
        ax.spines[spine].set_visible(True)
        ax.spines[spine].set_linewidth(1.0)
    ax.text(
        0.04,
        0.96,
        stats_text,
        transform=ax.transAxes,
        ha="left",
        va="top",
        fontsize=PLOT_FONT_SIZE - 1,
        fontweight="bold",
        fontstyle="normal",
        color="black",
    )
    for label in ax.get_xticklabels():
        label.set_fontweight("bold")
        label.set_fontsize(PLOT_FONT_SIZE)
        label.set_fontstyle("normal")
    for label in ax.get_yticklabels():
        label.set_fontweight("bold")
        label.set_fontsize(PLOT_FONT_SIZE)
        label.set_fontstyle("normal")
    plt.tight_layout()
    plt.savefig(plot_path, dpi=300, bbox_inches="tight")
    plt.close()
    return plot_path


def make_residual_plot(y_true: pd.Series, y_pred: np.ndarray, target: str, selector_model: str, best_model_name: str) -> Path:
    plot_path = RESIDUAL_PLOT_DIR / f"{target_to_safe_name(target)}_{selector_model}_{best_model_name}_residuals.png"
    residuals = np.asarray(y_true) - np.asarray(y_pred)
    plt.figure(figsize=(7, 6))
    plt.scatter(y_pred, residuals, alpha=0.8, edgecolors="none")
    plt.axhline(0, linestyle="--", color="gray", linewidth=1.5)
    plt.xlabel("Predicted")
    plt.ylabel("Residual (Observed - Predicted)")
    plt.title(f"Residual Plot: {target} [{selector_model} -> {best_model_name}]")
    plt.tight_layout()
    plt.savefig(plot_path, dpi=300, bbox_inches="tight")
    plt.close()
    return plot_path


def make_importance_bar_plot(importance_df: pd.DataFrame, target: str, selector_model: str, best_model_name: str) -> Path:
    plot_path = IMPORTANCE_PLOT_DIR / f"{target_to_safe_name(target)}_{selector_model}_{best_model_name}_importance.png"
    top_df = importance_df.sort_values("abs_importance", ascending=False).head(12).copy().iloc[::-1]
    n_bars = len(top_df)
    plt.rcParams.update({
        "font.family": PLOT_FONT_FAMILY,
        "font.size": PLOT_FONT_SIZE,
        "font.weight": "bold",
        "axes.labelsize": PLOT_FONT_SIZE,
        "axes.labelweight": "bold",
        "axes.titlesize": PLOT_FONT_SIZE,
        "axes.titleweight": "bold",
        "xtick.labelsize": PLOT_FONT_SIZE,
        "ytick.labelsize": PLOT_FONT_SIZE,
    })
    shap_like_cmap = LinearSegmentedColormap.from_list(
        "shap_like",
        ["#1E88E5", "#7B3FE4", "#FF0D57"]
    )
    colors = shap_like_cmap(np.linspace(0.1, 0.95, n_bars))
    fig, ax = plt.subplots(figsize=(10.5, 6.6), dpi=300)
    ax.barh(
        top_df["descriptor"],
        top_df["abs_importance"],
        color=colors,
        edgecolor="none",
        height=0.72,
    )
    ax.set_xlabel("Mean |SHAP Value|", fontweight="bold", fontsize=PLOT_FONT_SIZE)
    ax.set_title(
        f"Key Descriptors Driving {target_to_safe_name(target)} Prediction",
        pad=10,
        fontweight="bold",
        fontsize=PLOT_FONT_SIZE,
    )
    ax.grid(axis="x", color="#E6EEF5", linewidth=0.8)
    ax.set_axisbelow(True)
    for spine in ["top", "right", "left"]:
        ax.spines[spine].set_visible(False)
    ax.tick_params(axis="y", length=0)
    for label in ax.get_xticklabels():
        label.set_fontsize(PLOT_FONT_SIZE)
        label.set_fontweight("bold")
    for label in ax.get_yticklabels():
        label.set_fontsize(PLOT_FONT_SIZE)
        label.set_fontweight("bold")
    plt.tight_layout()
    plt.savefig(plot_path, dpi=300, bbox_inches="tight")
    plt.close()
    return plot_path


selector_catalog_df = pd.read_csv(STAGE1_DIR / "cell1_selector_feature_catalog.csv")
modeling_df = pd.read_csv(STAGE1_DIR / "cell1_descriptor_matrix.csv")
descriptor_label_map = build_descriptor_label_map(modeling_df["canonical_smiles"])
descriptor_columns = [c for c in modeling_df.columns if c not in BASE_COLUMNS]
performance_rows = []
model_comparison_rows = []
prediction_rows = []
importance_rows = []
shap_export_rows = []
summary_lines = [
    "# Cell 2 Predictive Modeling Summary",
    "",
    f"- Cell 1 selector catalog: `{STAGE1_DIR / 'cell1_selector_feature_catalog.csv'}`",
    f"- Optuna trials per model: {OPTUNA_TRIALS}",
    f"- Outer folds: {OUTER_SPLITS}",
    f"- Inner folds: {INNER_SPLITS}",
    f"- CV_n_jobs: {CV_N_JOBS}",
    "",
]

for target in TARGET_COLUMNS:
    print(f"[Cell2] ===== Processing target: {target} =====")
    target_df = modeling_df[["canonical_smiles", "CAS No.", "input_smiles", target] + descriptor_columns].copy()
    target_df = target_df[target_df[target].notna()].reset_index(drop=True)
    target_selector_rows = selector_catalog_df[selector_catalog_df["target"] == target].copy()

    for _, selector_row in target_selector_rows.iterrows():
        selector_model = selector_row["selector_model"]
        selected_features = [f.strip() for f in str(selector_row["selected_features"]).split(" | ") if f.strip()]
        selected_features = [f for f in selected_features if f in target_df.columns]
        if not selected_features:
            print(f"[Cell2] Skip {target} | selector={selector_model}: no valid saved features")
            continue
        print(f"[Cell2] {target} | selector={selector_model} | n_features={len(selected_features)}")
        print(f"[Cell2] {target} | selector={selector_model} | features={selected_features}")

        X_all = target_df[selected_features].copy()
        y_all = target_df[target].astype(float).reset_index(drop=True)
        outer_cv = KFold(n_splits=OUTER_SPLITS, shuffle=True, random_state=RANDOM_STATE)
        model_rows = []
        fold_artifacts = []

        for fold_idx, (dev_idx, test_idx) in enumerate(outer_cv.split(X_all), start=1):
            print(f"[Cell2] {target} | selector={selector_model} | outer fold {fold_idx}/{OUTER_SPLITS} (dev={len(dev_idx)}, test={len(test_idx)})")
            X_dev = X_all.iloc[dev_idx].reset_index(drop=True)
            y_dev = y_all.iloc[dev_idx].reset_index(drop=True)
            X_test = X_all.iloc[test_idx].reset_index(drop=True)
            y_test = y_all.iloc[test_idx].reset_index(drop=True)
            meta_test = target_df.iloc[test_idx].reset_index(drop=True)
            inner_cv = KFold(n_splits=INNER_SPLITS, shuffle=True, random_state=RANDOM_STATE + fold_idx)
            result_df, fitted_estimators = tune_and_compare_models(X_dev, y_dev, inner_cv=inner_cv, n_trials=OPTUNA_TRIALS)
            result_fold_df = result_df.copy()
            result_fold_df.insert(0, "outer_fold", fold_idx)
            result_fold_df.insert(0, "selector_model", selector_model)
            result_fold_df.insert(0, "target", target)
            result_fold_df["n_dev_samples"] = len(dev_idx)
            result_fold_df["n_test_samples"] = len(test_idx)
            result_fold_df["n_selected_features"] = len(selected_features)
            result_fold_df["selected_features"] = " | ".join(selected_features)
            result_fold_df["selected_features_readable"] = " | ".join(descriptor_label_map.get(f, f) for f in selected_features)
            result_fold_df["outer_test_r2"] = [r2_score(y_test, fitted_estimators[row["model_name"]].predict(X_test)) for _, row in result_df.iterrows()]
            result_fold_df["outer_test_rmse"] = [mean_squared_error(y_test, fitted_estimators[row["model_name"]].predict(X_test), squared=False) for _, row in result_df.iterrows()]
            result_fold_df["outer_test_mae"] = [mean_absolute_error(y_test, fitted_estimators[row["model_name"]].predict(X_test)) for _, row in result_df.iterrows()]
            model_rows.append(result_fold_df)
            fold_artifacts.append({"fold_idx": fold_idx, "X_test_selected": X_test, "y_test": y_test, "meta_test": meta_test, "fitted_estimators": fitted_estimators})

        model_comparison_df = pd.concat(model_rows, ignore_index=True)
        model_summary_df = (
            model_comparison_df.groupby("model_name", as_index=False)
            .agg(
                nested_cv_r2_mean=("outer_test_r2", "mean"),
                nested_cv_r2_std=("outer_test_r2", "std"),
                nested_cv_rmse_mean=("outer_test_rmse", "mean"),
                nested_cv_rmse_std=("outer_test_rmse", "std"),
                nested_cv_mae_mean=("outer_test_mae", "mean"),
                nested_cv_mae_std=("outer_test_mae", "std"),
                mean_inner_cv_r2=("cv_r2", "mean"),
                mean_inner_cv_rmse=("cv_rmse", "mean"),
                mean_inner_cv_mae=("cv_mae", "mean"),
            )
            .sort_values(["nested_cv_r2_mean", "nested_cv_rmse_mean"], ascending=[False, True])
            .reset_index(drop=True)
        )
        best_model_name = model_summary_df.iloc[0]["model_name"]
        print(f"[Cell2] {target} | selector={selector_model} | best predictive model={best_model_name}")

        prediction_rows_local = []
        shap_feature_frames = []
        shap_value_arrays = []
        for artifact in fold_artifacts:
            estimator = artifact["fitted_estimators"][best_model_name]
            X_test_selected = artifact["X_test_selected"]
            y_test = artifact["y_test"]
            meta_test = artifact["meta_test"]
            y_pred_test = estimator.predict(X_test_selected)
            shap_feature_df, shap_values = compute_shap_values(best_model_name, estimator, X_test_selected)
            shap_feature_frames.append(shap_feature_df)
            shap_value_arrays.append(shap_values)
            for row_idx in range(len(meta_test)):
                prediction_rows_local.append(
                    {
                        "target": target,
                        "selector_model": selector_model,
                        "best_model": best_model_name,
                        "outer_fold": artifact["fold_idx"],
                        "canonical_smiles": meta_test.loc[row_idx, "canonical_smiles"],
                        "CAS No.": meta_test.loc[row_idx, "CAS No."],
                        "input_smiles": meta_test.loc[row_idx, "input_smiles"],
                        "observed": float(y_test.iloc[row_idx]),
                        "predicted_outer_test": float(y_pred_test[row_idx]),
                    }
                )

        prediction_df = pd.DataFrame(prediction_rows_local)
        shap_feature_df_all, shap_values_all = aggregate_fold_shap_results(shap_feature_frames, shap_value_arrays)
        shap_metadata_df = prediction_df[[
            "outer_fold",
            "canonical_smiles",
            "CAS No.",
            "input_smiles",
            "observed",
            "predicted_outer_test",
        ]].copy()
        shap_export_df = build_shap_export_df(
            shap_feature_df_all,
            shap_values_all,
            shap_metadata_df,
            target,
            selector_model,
            best_model_name,
            descriptor_label_map,
        )
        importance_df = build_shap_importance_df(shap_feature_df_all, shap_values_all, label_map=descriptor_label_map)
        importance_df.insert(0, "best_model", best_model_name)
        importance_df.insert(0, "selector_model", selector_model)
        importance_df.insert(0, "target", target)

        shap_path = make_shap_plot_from_aggregated(shap_feature_df_all, shap_values_all, target, selector_model, best_model_name, descriptor_label_map)
        fit_path = make_fit_plot(
            prediction_df["observed"],
            prediction_df["predicted_outer_test"].to_numpy(),
            target,
            selector_model,
            best_model_name,
            stats_text=(
                f"$R^2$ = {model_summary_df.iloc[0]['nested_cv_r2_mean']:.3f}\n"
                f"MAE = {model_summary_df.iloc[0]['nested_cv_mae_mean']:.2f}"
            ),
        )
        residual_path = make_residual_plot(prediction_df["observed"], prediction_df["predicted_outer_test"].to_numpy(), target, selector_model, best_model_name)
        importance_path = make_importance_bar_plot(importance_df, target, selector_model, best_model_name)

        performance_rows.append(
            {
                "target": target,
                "selector_model": selector_model,
                "selector_best_step": selector_row["selector_best_step"],
                "selector_best_cv_r2": selector_row["selector_best_cv_r2"],
                "n_samples": len(target_df),
                "n_selected_features": len(selected_features),
                "selected_features": " | ".join(selected_features),
                "selected_features_readable": " | ".join(descriptor_label_map.get(f, f) for f in selected_features),
                "best_model": best_model_name,
                "nested_cv_r2_mean": model_summary_df.iloc[0]["nested_cv_r2_mean"],
                "nested_cv_r2_std": model_summary_df.iloc[0]["nested_cv_r2_std"],
                "nested_cv_rmse_mean": model_summary_df.iloc[0]["nested_cv_rmse_mean"],
                "nested_cv_rmse_std": model_summary_df.iloc[0]["nested_cv_rmse_std"],
                "nested_cv_mae_mean": model_summary_df.iloc[0]["nested_cv_mae_mean"],
                "nested_cv_mae_std": model_summary_df.iloc[0]["nested_cv_mae_std"],
                "mean_inner_cv_r2": model_summary_df.iloc[0]["mean_inner_cv_r2"],
                "mean_inner_cv_rmse": model_summary_df.iloc[0]["mean_inner_cv_rmse"],
                "mean_inner_cv_mae": model_summary_df.iloc[0]["mean_inner_cv_mae"],
                "shap_plot": str(shap_path),
                "fit_plot": str(fit_path),
                "residual_plot": str(residual_path),
                "importance_plot": str(importance_path),
            }
        )
        model_comparison_rows.append(model_comparison_df)
        prediction_rows.append(prediction_df)
        importance_rows.append(importance_df)
        shap_export_rows.append(shap_export_df)
        summary_lines.append(
            f"- `{target}` | selector=`{selector_model}` | best model=`{best_model_name}` | "
            f"outer-test R2={model_summary_df.iloc[0]['nested_cv_r2_mean']:.3f} +/- {model_summary_df.iloc[0]['nested_cv_r2_std']:.3f} | "
            f"MAE={model_summary_df.iloc[0]['nested_cv_mae_mean']:.3f} +/- {model_summary_df.iloc[0]['nested_cv_mae_std']:.3f}"
        )

performance_df = pd.DataFrame(performance_rows).sort_values(["target", "nested_cv_r2_mean"], ascending=[True, False])
model_comparison_df_all = pd.concat(model_comparison_rows, ignore_index=True)
prediction_df_all = pd.concat(prediction_rows, ignore_index=True)
importance_df_all = pd.concat(importance_rows, ignore_index=True)
shap_export_df_all = pd.concat(shap_export_rows, ignore_index=True)

performance_df.to_csv(STAGE2_DIR / "cell2_predictive_model_performance.csv", index=False)
model_comparison_df_all.to_csv(STAGE2_DIR / "cell2_all_model_comparisons.csv", index=False)
prediction_df_all.to_csv(STAGE2_DIR / "cell2_predictions.csv", index=False)
importance_df_all.to_csv(STAGE2_DIR / "cell2_importance.csv", index=False)
shap_export_df_all.to_csv(STAGE2_DIR / "cell2_shap_values.csv", index=False)

with pd.ExcelWriter(STAGE2_DIR / "cell2_predictive_model_results.xlsx", engine="openpyxl") as writer:
    performance_df.to_excel(writer, sheet_name="performance", index=False)
    model_comparison_df_all.to_excel(writer, sheet_name="all_model_comparisons", index=False)
    prediction_df_all.to_excel(writer, sheet_name="predictions", index=False)
    importance_df_all.to_excel(writer, sheet_name="importance", index=False)
    shap_export_df_all.to_excel(writer, sheet_name="shap_values", index=False)

(STAGE2_DIR / "cell2_predictive_model_summary.md").write_text("\n".join(summary_lines), encoding="utf-8")
print(f"[Cell2] Done. Outputs written to {STAGE2_DIR}")


[15:43:37] DEPRECATION WARNING: please use MorganGenerator
[15:43:37] DEPRECATION WARNING: please use MorganGenerator
[15:43:37] DEPRECATION WARNING: please use MorganGenerator
[15:43:37] DEPRECATION WARNING: please use MorganGenerator
[15:43:37] DEPRECATION WARNING: please use MorganGenerator
[15:43:37] DEPRECATION WARNING: please use MorganGenerator
[15:43:37] DEPRECATION WARNING: please use MorganGenerator
[15:43:37] DEPRECATION WARNING: please use MorganGenerator
[15:43:37] DEPRECATION WARNING: please use MorganGenerator
[15:43:37] DEPRECATION WARNING: please use MorganGenerator
[15:43:37] DEPRECATION WARNING: please use MorganGenerator
[15:43:37] DEPRECATION WARNING: please use MorganGenerator
[15:43:37] DEPRECATION WARNING: please use MorganGenerator
[15:43:37] DEPRECATION WARNING: please use MorganGenerator
[15:43:37] DEPRECATION WARNING: please use MorganGenerator
[15:43:37] DEPRECATION WARNING: please use MorganGenerator
[15:43:37] DEPRECATION WARNING: please use MorganGenerat

[Cell2] ===== Processing target: FEaverage H2 =====
[Cell2] FEaverage H2 | selector=ridge | n_features=31
[Cell2] FEaverage H2 | selector=ridge | features=['morgan_bit_37', 'morgan_bit_108', 'fr_para_hydroxylation', 'morgan_bit_22', 'morgan_bit_41', 'morgan_bit_31', 'morgan_bit_72', 'morgan_bit_14', 'morgan_bit_69', 'morgan_bit_11', 'BalabanJ', 'BCUT2D_MRLOW', 'morgan_bit_50', 'morgan_bit_83', 'morgan_bit_73', 'morgan_bit_0', 'SMR_VSA10', 'morgan_bit_51', 'morgan_bit_76', 'fr_Ndealkylation1', 'electronic_n_charge_max', 'PEOE_VSA3', 'morgan_bit_61', 'VSA_EState5', 'morgan_bit_59', 'RingCount', 'fr_allylic_oxid', 'SlogP_VSA10', 'morgan_bit_102', 'morgan_bit_35', 'morgan_bit_117']
[Cell2] FEaverage H2 | selector=ridge | outer fold 1/5 (dev=88, test=22)
  [Model Search] Start comparing 13 models on 31 selected features
    [Optuna] Start tuning model: ridge with 30 trials


[I 2026-06-12 15:43:40,163] Trial 0 finished with value: -0.22699060768947632 and parameters: {'model__alpha': 0.31489116479568624}. Best is trial 0 with value: -0.22699060768947632.
[I 2026-06-12 15:43:40,715] Trial 1 finished with value: -0.13574302625977885 and parameters: {'model__alpha': 63.512210106407046}. Best is trial 1 with value: -0.13574302625977885.
[I 2026-06-12 15:43:41,529] Trial 2 finished with value: -0.09226895778970409 and parameters: {'model__alpha': 8.471801418819979}. Best is trial 2 with value: -0.09226895778970409.
[I 2026-06-12 15:43:42,070] Trial 3 finished with value: -0.1137125817798809 and parameters: {'model__alpha': 2.481040974867813}. Best is trial 2 with value: -0.09226895778970409.
[I 2026-06-12 15:43:42,600] Trial 4 finished with value: -0.2718321425291843 and parameters: {'model__alpha': 0.04207988669606638}. Best is trial 2 with value: -0.09226895778970409.
[I 2026-06-12 15:43:43,356] Trial 5 finished with value: -0.27183405358999657 and parameters

  [Model Search] ridge: CV R2=0.2666, RMSE=12.1529, MAE=8.2851
    [Optuna] Start tuning model: lasso with 30 trials


[I 2026-06-12 15:43:45,261] Trial 4 finished with value: -0.26731680830260107 and parameters: {'model__alpha': 0.0029380279387035343}. Best is trial 3 with value: -0.1483334069784408.
[I 2026-06-12 15:43:45,306] Trial 5 finished with value: -0.2673189023458583 and parameters: {'model__alpha': 0.0029375384576328283}. Best is trial 3 with value: -0.1483334069784408.
[I 2026-06-12 15:43:45,360] Trial 6 finished with value: -0.2738489833279848 and parameters: {'model__alpha': 0.0014936568554617625}. Best is trial 3 with value: -0.1483334069784408.
[I 2026-06-12 15:43:45,404] Trial 7 finished with value: -0.19978875017127068 and parameters: {'model__alpha': 0.39676050770529875}. Best is trial 3 with value: -0.1483334069784408.
[I 2026-06-12 15:43:45,449] Trial 8 finished with value: -0.14741451722747279 and parameters: {'model__alpha': 0.06358358856676251}. Best is trial 8 with value: -0.14741451722747279.
[I 2026-06-12 15:43:45,499] Trial 9 finished with value: -0.1343778108333451 and para

  [Model Search] lasso: CV R2=0.2484, RMSE=12.3028, MAE=8.4367
    [Optuna] Start tuning model: elastic_net with 30 trials


... (115 outputs left out)

  [Model Search] pls: CV R2=0.2810, RMSE=12.1125, MAE=8.0564
    [Optuna] Start tuning model: svr_rbf with 30 trials


[I 2026-06-12 15:49:02,689] Trial 3 finished with value: 0.013148972845383744 and parameters: {'model__C': 13.311216080736884, 'model__gamma': 0.00115279871282324, 'model__epsilon': 0.44447541666908114}. Best is trial 1 with value: 0.014112696173296645.
[I 2026-06-12 15:49:02,744] Trial 4 finished with value: 0.23122541702585622 and parameters: {'model__C': 31.428808908401084, 'model__gamma': 0.004335281794951566, 'model__epsilon': 0.020366442026830918}. Best is trial 4 with value: 0.23122541702585622.
[I 2026-06-12 15:49:02,790] Trial 5 finished with value: -0.08975252432225415 and parameters: {'model__C': 0.35498788321965025, 'model__gamma': 0.008179499475211672, 'model__epsilon': 0.0779014312627624}. Best is trial 4 with value: 0.23122541702585622.
[I 2026-06-12 15:49:02,835] Trial 6 finished with value: -0.019841671042395825 and parameters: {'model__C': 1.976218934028007, 'model__gamma': 0.007476312062252299, 'model__epsilon': 0.10952662748632554}. Best is trial 4 with value: 0.231

  [Model Search] svr_rbf: CV R2=0.2411, RMSE=12.4443, MAE=7.0764
    [Optuna] Start tuning model: knn with 30 trials


[I 2026-06-12 15:49:04,573] Trial 4 finished with value: -0.21549010901284765 and parameters: {'model__n_neighbors': 11, 'model__weights': 'distance', 'model__p': 2}. Best is trial 0 with value: 0.07170863288808342.
[I 2026-06-12 15:49:04,618] Trial 5 finished with value: -0.2586026131805199 and parameters: {'model__n_neighbors': 13, 'model__weights': 'distance', 'model__p': 1}. Best is trial 0 with value: 0.07170863288808342.
[I 2026-06-12 15:49:04,663] Trial 6 finished with value: 0.054245601819608226 and parameters: {'model__n_neighbors': 11, 'model__weights': 'uniform', 'model__p': 2}. Best is trial 0 with value: 0.07170863288808342.
[I 2026-06-12 15:49:04,721] Trial 7 finished with value: -0.0024408230104838945 and parameters: {'model__n_neighbors': 13, 'model__weights': 'uniform', 'model__p': 1}. Best is trial 0 with value: 0.07170863288808342.
[I 2026-06-12 15:49:04,766] Trial 8 finished with value: -0.25820783725809937 and parameters: {'model__n_neighbors': 3, 'model__weights':